# HUAN LUYEN MO HINH REINFORCEMENT LEARNING (PPO) - BAN DO THANH PHO 60x60

Notebook nay huan luyen mang no-ron PPO (Proximal Policy Optimization) cho Be Bo trong Ban Do Do Thi 60x60.

### Thong So Ky Thuat:
- **Moi truong**: `ChildWorldEnv` (Thanh pho 60x60m, 14 vat the vat ly do thi).
- **Vector quan sat (Observation)**: 28 chieu lien tuc:
  - `robot_pos` (2)
  - `goal_pos` (2)
  - `food_pos` (2)
  - `water_pos` (2)
  - `bed_pos` (2)
  - `obstacles` (12 chieu: 6 vat the do thi gan nhat * 2)
  - `internal_vec` (6 chieu: Ty le nang luong Kcal, Hunger, Thirst, Fatigue, Pain, Curiosity)
- **Vector hanh dong (Action)**: 2 chieu van toc lien tuc `[vx, vz]` trong khoang [-2.4, 2.4] m/s.
- **Tuong thich**: Mo hinh xuat ra file `obstacle_ppo_latest.zip` tuong thich 100% plug-and-play voi backend FastAPI va giao dien 3D Three.js.

In [ ]:
# [1] CAI DAT CAC THU VIEN HO TRO
!pip install -q "stable-baselines3[extra]" gymnasium matplotlib tqdm

In [ ]:
# [2] DINH NGHIA HE THONG NANG LUONG SINH HOC VA NHU CAU NOI TAI (ENERGY & DRIVES)
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from typing import Dict, Any, Optional, Tuple

class EnergySystem:
    def __init__(self, energy: float = 800.0, energy_max: float = 1000.0, basal_rate: float = 1.2):
        self.energy_max: float = float(energy_max)
        self.energy: float = float(np.clip(energy, 0.0, self.energy_max))
        self.digesting: float = 0.0
        self.fatigue: float = 0.0
        self.basal_rate: float = float(basal_rate)
        self.motor_cost_coeff: float = 3.5
        self.cognitive_cost_coeff: float = 1.0
        self.stress_cost_coeff: float = 1.5
        self.digestion_rate: float = 18.0

    @property
    def energy_ratio(self) -> float:
        return float(np.clip(self.energy / max(self.energy_max, 1.0), 0.0, 1.0))

    @property
    def alpha_motor(self) -> float:
        energy_factor = np.clip(self.energy / (0.3 * self.energy_max), 0.2, 1.0)
        fatigue_factor = 1.0 - 0.55 * self.fatigue
        return float(np.clip(energy_factor * fatigue_factor, 0.2, 1.0))

    def step(self, dt: float, motor_force_norm: float = 0.0, cognitive_load: float = 0.1,
             stress_level: float = 0.0, is_resting: bool = False) -> Dict[str, float]:
        if is_resting:
            self.energy = min(self.energy_max, self.energy - 0.3 * self.basal_rate * dt)
            self.fatigue = max(0.0, self.fatigue - 0.05 * dt)
        else:
            cost = (self.basal_rate
                    + self.motor_cost_coeff * motor_force_norm
                    + self.cognitive_cost_coeff * cognitive_load
                    + self.stress_cost_coeff * stress_level) * dt
            self.energy = max(0.0, self.energy - cost)
            self.fatigue = min(1.0, self.fatigue + 0.003 * dt * (1.0 + motor_force_norm))

        if self.digesting > 0.0:
            absorbed = min(self.digesting, self.digestion_rate * dt)
            self.digesting -= absorbed
            self.energy = min(self.energy_max, self.energy + absorbed)

        return {"energy": self.energy, "energy_ratio": self.energy_ratio, "fatigue": self.fatigue}

    def eat(self, kcal: float = 120.0):
        self.digesting = min(400.0, self.digesting + kcal)

    def rest(self, duration_sec: float = 2.0):
        self.fatigue = max(0.0, self.fatigue - 0.12 * duration_sec)


class DriveSystem:
    DRIVE_NAMES = ("hunger", "thirst", "fatigue", "pain", "fear", "curiosity", "comfort", "social")

    def __init__(self):
        self.drives: Dict[str, float] = {
            "hunger": 0.20,
            "thirst": 0.15,
            "fatigue": 0.10,
            "pain": 0.00,
            "fear": 0.00,
            "curiosity": 0.55,
            "comfort": 0.85,
            "social": 0.30,
        }

    def step(self, dt: float, energy_sys: EnergySystem, events: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        events = events or {}
        hunger_drift = 0.0035 * dt
        if energy_sys.energy_ratio < 0.4:
            hunger_drift += (0.4 - energy_sys.energy_ratio) * 0.02 * dt
        self.drives["hunger"] = float(np.clip(self.drives["hunger"] + hunger_drift, 0.0, 1.0))
        if events.get("ate_food", False):
            self.drives["hunger"] = max(0.0, self.drives["hunger"] - 0.40)

        self.drives["thirst"] = float(np.clip(self.drives["thirst"] + 0.0045 * dt, 0.0, 1.0))
        if events.get("drank_water", False):
            self.drives["thirst"] = max(0.0, self.drives["thirst"] - 0.50)

        self.drives["fatigue"] = energy_sys.fatigue
        if events.get("is_resting", False):
            self.drives["fatigue"] = max(0.0, self.drives["fatigue"] - 0.08 * dt)

        if events.get("collided", False):
            self.drives["pain"] = min(1.0, self.drives["pain"] + 0.60)
            self.drives["fear"] = min(1.0, self.drives["fear"] + 0.45)
        else:
            self.drives["pain"] = max(0.0, self.drives["pain"] - 0.15 * dt)
            self.drives["fear"] = max(0.0, self.drives["fear"] - 0.10 * dt)

        self.drives["curiosity"] = float(np.clip(self.drives["curiosity"] + 0.002 * dt, 0.0, 1.0))
        return {"drives": dict(self.drives)}

print("[OK] Da dinh nghia EnergySystem & DriveSystem thanh cong.")

In [ ]:
# [3] MOI TRUONG GYMNASIUM BAN DO THANH PHO 60x60 (CHILDE WORLD ENV)
class ChildWorldEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, is_gym_training: bool = True):
        super().__init__()
        self.world_size: float = 60.0
        self.robot_radius: float = 0.45
        self.goal_radius: float = 1.20
        self.zone_radius: float = 1.60
        self.max_speed: float = 2.4
        self.dt: float = 0.05
        self.is_gym_training: bool = is_gym_training

        # Toa do cac khu vuc trong do thi 60x60
        self.bed_pos = np.array([8.0, 8.0], dtype=np.float32)
        self.pool_pos = np.array([45.0, 14.0], dtype=np.float32)
        self.park_pos = np.array([14.0, 44.0], dtype=np.float32)
        self.water_pos = np.array([39.0, 8.0], dtype=np.float32)
        self.food_pos = np.array([46.0, 44.0], dtype=np.float32)
        self.goal_pos = np.array([10.0, 52.0], dtype=np.float32)
        self.robot_pos = np.array([8.0, 14.0], dtype=np.float32)

        # 14 vat the vat ly do thi (Nha cua, xe co, ho boi, cong vien)
        self.city_obstacles = np.array([
            [12.0, 12.0],  # Villa
            [24.0, 10.0],  # Mini Mart
            [10.0, 24.0],  # Chung cu cao tang
            [24.0, 24.0],  # Nha pho lien ke
            [30.0, 16.0],  # Sedan do
            [46.0, 36.0],  # SUV xanh
            [52.0, 44.0],  # Xe Van giao hang
            [16.0, 32.0],  # Taxi vang
            [45.0, 14.0],  # Mep bo ho boi
            [53.0, 8.0],   # Choi cuu ho
            [14.0, 44.0],  # Cau truot cong vien
            [14.0, 52.0],  # Bon hoa da
            [30.0, 30.0],  # Bot giao lo
            [48.0, 50.0],  # Thap dong ho
        ], dtype=np.float32)

        self.city_obstacle_radii = np.array([
            3.4, 2.8, 3.2, 2.6, 1.8, 1.9, 2.2, 1.8, 3.8, 1.6, 2.4, 2.0, 1.6, 2.2
        ], dtype=np.float32)

        self.num_obstacles = 6
        self.obstacles = self.city_obstacles[:6]

        self.action_space = spaces.Box(
            low=-self.max_speed, high=self.max_speed, shape=(2,), dtype=np.float32
        )

        # 28 chieu quan sat: robot(2) + goal(2) + food(2) + water(2) + bed(2) + obstacles(12) + internal(6)
        obs_dim = 2 + 2 + 2 + 2 + 2 + (self.num_obstacles * 2) + 6
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )

        self.energy_sys = EnergySystem(energy=800.0, energy_max=1000.0)
        self.drive_sys = DriveSystem()
        self.episode_steps: int = 0
        self.max_episode_steps: int = 500

    def get_safe_random_pos(self, min_wall_dist: float = 3.5, min_obs_dist: float = 2.0,
                            avoid_points: Optional[list] = None, min_avoid_dist: float = 5.0) -> np.ndarray:
        avoid_points = avoid_points or []
        for _ in range(300):
            rx = float(np.random.uniform(min_wall_dist, self.world_size - min_wall_dist))
            rz = float(np.random.uniform(min_wall_dist, self.world_size - min_wall_dist))
            candidate = np.array([rx, rz], dtype=np.float32)

            obs_safe = True
            for obs, rad in zip(self.city_obstacles, self.city_obstacle_radii):
                if float(np.linalg.norm(candidate - obs)) < (rad + min_obs_dist):
                    obs_safe = False
                    break
            if not obs_safe:
                continue

            avoid_safe = True
            for pt in avoid_points:
                if pt is not None and float(np.linalg.norm(candidate - np.asarray(pt, dtype=np.float32))) < min_avoid_dist:
                    avoid_safe = False
                    break
            if not avoid_safe:
                continue

            return np.array([round(rx, 2), round(rz, 2)], dtype=np.float32)
        return np.array([float(np.random.uniform(5.0, 55.0)), float(np.random.uniform(5.0, 55.0))], dtype=np.float32)

    def reset(self, seed: Optional[int] = None, options: Optional[Dict[str, Any]] = None) -> Tuple[np.ndarray, Dict[str, Any]]:
        super().reset(seed=seed)
        self.robot_pos = self.get_safe_random_pos(min_wall_dist=4.0, min_obs_dist=2.2)
        self.goal_pos = self.get_safe_random_pos(
            min_wall_dist=4.0, min_obs_dist=2.5,
            avoid_points=[self.robot_pos], min_avoid_dist=8.0
        )
        self.episode_steps = 0
        self.energy_sys = EnergySystem(energy=800.0, energy_max=1000.0)
        self.drive_sys = DriveSystem()
        return self._get_obs(), {}

    def _get_obs(self) -> np.ndarray:
        internal_vec = np.array([
            self.energy_sys.energy_ratio,
            self.drive_sys.drives["hunger"],
            self.drive_sys.drives["thirst"],
            self.drive_sys.drives["fatigue"],
            self.drive_sys.drives["pain"],
            self.drive_sys.drives["curiosity"],
        ], dtype=np.float32)

        dists = np.linalg.norm(self.city_obstacles - self.robot_pos, axis=1)
        closest_indices = np.argsort(dists)[:6]
        self.obstacles = self.city_obstacles[closest_indices]

        return np.concatenate([
            self.robot_pos,
            self.goal_pos,
            self.food_pos,
            self.water_pos,
            self.bed_pos,
            self.obstacles.flatten(),
            internal_vec,
        ]).astype(np.float32)

    def step(self, action: np.ndarray) -> Tuple[np.ndarray, float, bool, bool, Dict[str, Any]]:
        self.episode_steps += 1
        act = np.clip(np.asarray(action, dtype=np.float32), -self.max_speed, self.max_speed)
        old_dist = float(np.linalg.norm(self.robot_pos - self.goal_pos))

        alpha_m = self.energy_sys.alpha_motor
        eff_act = act * alpha_m
        force_norm = float(np.linalg.norm(eff_act) / self.max_speed)

        self.robot_pos = self.robot_pos + eff_act * self.dt
        self.robot_pos = np.clip(self.robot_pos, 0.5, self.world_size - 0.5)

        dist_to_goal = float(np.linalg.norm(self.robot_pos - self.goal_pos))
        dist_to_food = float(np.linalg.norm(self.robot_pos - self.food_pos))
        dist_to_water = float(np.linalg.norm(self.robot_pos - self.water_pos))
        dist_to_bed = float(np.linalg.norm(self.robot_pos - self.bed_pos))

        collided = False
        hazard_prox = 0.0
        for obs, rad in zip(self.city_obstacles, self.city_obstacle_radii):
            diff = self.robot_pos - obs
            dist = float(np.linalg.norm(diff))
            thresh = self.robot_radius + rad
            if dist < thresh:
                collided = True
                if dist > 0.001:
                    normal = diff / dist
                    self.robot_pos = obs + normal * (thresh + 0.12)
                    self.robot_pos = np.clip(self.robot_pos, 1.2, self.world_size - 1.2)
            if dist < (thresh + 2.5):
                hazard_prox = max(hazard_prox, (thresh + 2.5 - dist) / 2.5)

        # Kiem tra va cham tuong bien
        wall_thresh = 0.9
        if self.robot_pos[0] < wall_thresh:
            collided = True
            self.robot_pos[0] = 1.25
        elif self.robot_pos[0] > (self.world_size - wall_thresh):
            collided = True
            self.robot_pos[0] = self.world_size - 1.25

        if self.robot_pos[1] < wall_thresh:
            collided = True
            self.robot_pos[1] = 1.25
        elif self.robot_pos[1] > (self.world_size - wall_thresh):
            collided = True
            self.robot_pos[1] = self.world_size - 1.25

        ate_food = dist_to_food < self.zone_radius
        drank_water = dist_to_water < self.zone_radius
        is_resting = dist_to_bed < self.zone_radius and force_norm < 0.25

        if ate_food:
            self.energy_sys.eat(kcal=35.0)
        self.energy_sys.step(dt=self.dt, motor_force_norm=force_norm, cognitive_load=0.15,
                             stress_level=self.drive_sys.drives["fear"], is_resting=is_resting)

        events = {
            "collided": collided,
            "hazard_proximity": hazard_prox,
            "ate_food": ate_food,
            "drank_water": drank_water,
            "is_resting": is_resting,
        }
        self.drive_sys.step(dt=self.dt, energy_sys=self.energy_sys, events=events)

        progress = old_dist - dist_to_goal
        reward = (progress * 18.0) - 0.08 - (dist_to_goal / self.world_size) * 0.10 - 0.02 * force_norm

        terminated = False
        if ate_food and self.drive_sys.drives["hunger"] > 0.25:
            reward += 6.0
        if drank_water and self.drive_sys.drives["thirst"] > 0.25:
            reward += 5.0
        if is_resting and self.drive_sys.drives["fatigue"] > 0.4:
            reward += 3.5

        if collided:
            reward -= 50.0
            if self.is_gym_training:
                terminated = True
        elif dist_to_goal < self.goal_radius:
            reward += 150.0
            terminated = True

        truncated = self.episode_steps >= self.max_episode_steps
        info = {
            "dist_to_goal": dist_to_goal,
            "collided": collided,
            "robot": self.robot_pos.tolist(),
            "goal": self.goal_pos.tolist(),
            "obstacles": self.obstacles.tolist(),
        }
        return self._get_obs(), float(reward), terminated, truncated, info

print("[OK] Da khoi tao ChildWorldEnv (28 chieu quan sat, 60x60m). Khoi tao thanh cong.")

In [ ]:
# [4] HUAN LUYEN PPO VOI 8 MOI TRUONG SONG SONG (VECTORIZED RL)
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

NUM_ENVS = 8
TOTAL_STEPS = 150_000
MODEL_NAME = "obstacle_ppo_latest"

print(f"[INFO] Dang tao {NUM_ENVS} moi truong song song tren GPU/CPU Colab...")
vec_env = make_vec_env(ChildWorldEnv, n_envs=NUM_ENVS)

print("[INFO] Khoi tao mo hinh PPO (Actor-Critic Multi-Layer Perceptron)...\n")
model = PPO(
    "MlpPolicy",
    vec_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.005,
    verbose=1,
    device="auto",
)

print(f"[INFO] Bat dau huan luyen {TOTAL_STEPS:,} buoc thoi gian...")
try:
    model.learn(total_timesteps=TOTAL_STEPS, progress_bar=True)
except Exception:
    model.learn(total_timesteps=TOTAL_STEPS, progress_bar=False)

model.save(MODEL_NAME)
print(f"\n[THANH CONG] Mo hinh da duoc huan luyen va luu thanh: {MODEL_NAME}.zip")

In [ ]:
# [5] DANH GIA VA SO SANH: CHUA HUAN LUYEN (RANDOM) VS PPO (TRAINED)
def evaluate_performance(model_path="obstacle_ppo_latest", num_episodes=50):
    env = ChildWorldEnv(is_gym_training=False)
    results = {}

    test_cases = [
        ("Chua Huan Luyen (Random Agent)", None),
        ("Mo Hinh PPO 150k Steps (Toi Uu)", PPO.load(model_path))
    ]

    for label, policy in test_cases:
        success_count = 0
        collision_count = 0
        step_list = []
        reward_list = []

        for _ in range(num_episodes):
            obs, _ = env.reset()
            done = False
            ep_reward = 0.0
            ep_steps = 0

            while not done and ep_steps < 300:
                if policy is not None:
                    action, _ = policy.predict(obs, deterministic=True)
                else:
                    action = env.action_space.sample()

                obs, r, term, trunc, info = env.step(action)
                ep_reward += r
                ep_steps += 1

                if term or trunc:
                    if info.get("collided", False):
                        collision_count += 1
                    elif info.get("dist_to_goal", 99) < env.goal_radius:
                        success_count += 1
                    done = True

            step_list.append(ep_steps)
            reward_list.append(ep_reward)

        results[label] = {
            "success_rate": (success_count / num_episodes) * 100.0,
            "collision_rate": (collision_count / num_episodes) * 100.0,
            "avg_steps": np.mean(step_list),
            "avg_reward": np.mean(reward_list),
        }

    print(f"\n{'='*88}")
    print(f"BANG SO SANH CHAT LUONG DIEU KHIEN ({num_episodes} EPISODES)")
    print(f"{'='*88}")
    print(f"{'Mo Hinh':<35} | {'Thanh Cong (%)':<16} | {'Va Cham (%)':<14} | {'Buoc TB':<10} | {'Reward TB':<10}")
    print(f"{'-'*88}")
    for name, m in results.items():
        print(f"{name:<35} | {m['success_rate']:<16.1f} | {m['collision_rate']:<14.1f} | {m['avg_steps']:<10.1f} | {m['avg_reward']:<10.1f}")
    print(f"{'='*88}\n")

evaluate_performance(model_path=MODEL_NAME, num_episodes=50)

In [ ]:
# [6] TRUC QUAN HOA QUY DAO DI CHUYEN NE VAT CAN TREN BAN DO 60x60
import matplotlib.pyplot as plt

def plot_city_trajectory(model_path="obstacle_ppo_latest"):
    env = ChildWorldEnv(is_gym_training=False)
    obs, _ = env.reset()
    policy = PPO.load(model_path)

    trajectory = [env.robot_pos.copy()]
    done = False
    steps = 0

    while not done and steps < 350:
        action, _ = policy.predict(obs, deterministic=True)
        obs, r, term, trunc, info = env.step(action)
        trajectory.append(env.robot_pos.copy())
        steps += 1
        if term or trunc:
            done = True

    traj = np.array(trajectory)
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_xlim(0, env.world_size)
    ax.set_ylim(0, env.world_size)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.4)

    # 14 vat the do thi
    for i, (obs_coord, rad) in enumerate(zip(env.city_obstacles, env.city_obstacle_radii)):
        circle = plt.Circle(obs_coord, rad, color='red', alpha=0.35, label='Vat The Do Thi' if i == 0 else '')
        ax.add_patch(circle)
        ax.plot(obs_coord[0], obs_coord[1], 'rx')

    # Goal
    goal_c = plt.Circle(env.goal_pos, env.goal_radius, color='green', alpha=0.35, label='Khu Vuc Dich')
    ax.add_patch(goal_c)
    ax.plot(env.goal_pos[0], env.goal_pos[1], 'g*', markersize=14, label='Dich Den')

    # Xuat phat va duong di
    ax.plot(traj[0, 0], traj[0, 1], 'bs', markersize=9, label='Diem Xuat Phat')
    ax.plot(traj[:, 0], traj[:, 1], 'b-', linewidth=2.5, alpha=0.85, label='Quy Dao Be Bo')
    ax.plot(traj[-1, 0], traj[-1, 1], 'go' if info.get('dist_to_goal', 99) < env.goal_radius else 'ro',
            markersize=9, label='Diem Ket Thuc')

    plt.title(f"QUY DAO DI CHUYEN TRONG THANH PHO 60x60 ({steps} buoc, Dist to Goal: {info['dist_to_goal']:.2f}m)")
    plt.xlabel("Toa do X (met)")
    plt.ylabel("Toa do Z (met)")
    plt.legend(loc='upper right')
    plt.show()

plot_city_trajectory(model_path=MODEL_NAME)

In [ ]:
# [7] TAI FILE MODEL VE MAY TINH DE NAP VAO UNG DUNG LOCAL
from google.colab import files
import os

target_file = f"{MODEL_NAME}.zip"
if os.path.exists(target_file):
    print(f"[INFO] Dang chuan bi tai file: {target_file} ve may tinh...")
    files.download(target_file)
    print("[THONG BAO] File dang duoc trinh duyet tu dong tai xuong.")
    print(f"[HUONG DAN] Hay luu/chep file '{target_file}' vao thu muc:\nbackend/models/{target_file}")
else:
    print(f"[LOI] Khong tim thay file {target_file}. Hay chay lai cell train.")